### Установка PySpark

In [2]:
!pip install -q pyspark

### Создание spsrk-сессии и установка зависимости

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Titanic').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/23 16:22:33 WARN Utils: Your hostname, Artems-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.149 instead (on interface en0)
26/09/23 16:22:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/opt/anaconda3/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/09/23 16:22:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Импортирование дополнительных библиотек

In [5]:
from itertools import chain
from pyspark.sql.functions import count, mean, when, lit, create_map, regexp_extract

### Импортирование данных

In [6]:
df1 = spark.read.csv('tit_train.csv', header=True, inferSchema=True)

### Просмотр схемы

In [7]:
df1.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [8]:
df1.show(3)

+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|PassengerId|Survived|Pclass|                Name|   Sex| Age|SibSp|Parch|          Ticket|   Fare|Cabin|Embarked|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
|          1|       0|     3|Braund, Mr. Owen ...|  male|22.0|    1|    0|       A/5 21171|   7.25| NULL|       S|
|          2|       1|     1|Cumings, Mrs. Joh...|female|38.0|    1|    0|        PC 17599|71.2833|  C85|       C|
|          3|       1|     3|Heikkinen, Miss. ...|female|26.0|    0|    0|STON/O2. 3101282|  7.925| NULL|       S|
+-----------+--------+------+--------------------+------+----+-----+-----+----------------+-------+-----+--------+
only showing top 3 rows


### Отображение в Pandas

In [9]:
df1.limit(20).toPandas()

/opt/anaconda3/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


### Выборки

In [11]:
df1.select('Survived', 'Pclass', 'Age', 'Fare').show(5)

+--------+------+----+-------+
|Survived|Pclass| Age|   Fare|
+--------+------+----+-------+
|       0|     3|22.0|   7.25|
|       1|     1|38.0|71.2833|
|       1|     3|26.0|  7.925|
|       1|     1|35.0|   53.1|
|       0|     3|35.0|   8.05|
+--------+------+----+-------+
only showing top 5 rows


### Описательная статистика

In [12]:
df1.select('Survived', 'Pclass', 'Age', 'Fare').summary().show()

26/09/23 16:50:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------------+------------------+------------------+-----------------+
|summary|           Survived|            Pclass|               Age|             Fare|
+-------+-------------------+------------------+------------------+-----------------+
|  count|                891|               891|               714|              891|
|   mean| 0.3838383838383838| 2.308641975308642| 29.69911764705882| 32.2042079685746|
| stddev|0.48659245426485753|0.8360712409770491|14.526497332334035|49.69342859718089|
|    min|                  0|                 1|              0.42|              0.0|
|    25%|                  0|                 2|              20.0|           7.8958|
|    50%|                  0|                 3|              28.0|          14.4542|
|    75%|                  1|                 3|              38.0|             31.0|
|    max|                  1|                 3|              80.0|         512.3292|
+-------+-------------------+------------------+------

In [13]:
print('Количество записей: \t', df1.count())
print('Количество переменных: \t', len(df1.columns))

Количество записей: 	 891
Количество переменных: 	 12


### EDA

### Сколько человек выжило

In [14]:
df1.groupBy('Survived').count().show()

+--------+-----+
|Survived|count|
+--------+-----+
|       1|  342|
|       0|  549|
+--------+-----+



### Количественные характеристики

In [15]:
df1.groupBy('Survived').mean('Fare', 'Age').show()

+--------+------------------+------------------+
|Survived|         avg(Fare)|          avg(Age)|
+--------+------------------+------------------+
|       1| 48.39540760233917|28.343689655172415|
|       0|22.117886885245877| 30.62617924528302|
+--------+------------------+------------------+



### Факторные переменные

In [16]:
df1.groupBy('Survived').pivot('Sex').count().show()

+--------+------+----+
|Survived|female|male|
+--------+------+----+
|       1|   233| 109|
|       0|    81| 468|
+--------+------+----+



In [17]:
df1.groupBy('Survived').pivot('Pclass').count().show()

+--------+---+---+---+
|Survived|  1|  2|  3|
+--------+---+---+---+
|       1|136| 87|119|
|       0| 80| 97|372|
+--------+---+---+---+



In [18]:
df1.groupBy('Survived').pivot('SibSp').count().show()

+--------+---+---+---+---+---+----+----+
|Survived|  0|  1|  2|  3|  4|   5|   8|
+--------+---+---+---+---+---+----+----+
|       1|210|112| 13|  4|  3|NULL|NULL|
|       0|398| 97| 15| 12| 15|   5|   7|
+--------+---+---+---+---+---+----+----+



### Извлечение признаков

In [19]:
for col in df1.columns:
    print(col.ljust(20), df1.filter(df1[col].isNull()).count())

PassengerId          0
Survived             0
Pclass               0
Name                 0
Sex                  0
Age                  177
SibSp                0
Parch                0
Ticket               0
Fare                 0
Cabin                687
Embarked             2


### Извлечение данных о титулах

In [20]:
df1 = df1.withColumn('Title', regexp_extract(df1['Name'], '([A-Za-z]+)\.', 1))
df1.groupBy('Title').agg(count('Age'), mean('Age')).sort('count(Age)').show()

+--------+----------+------------------+
|   Title|count(Age)|          avg(Age)|
+--------+----------+------------------+
|     Don|         1|              40.0|
|Countess|         1|              33.0|
|    Lady|         1|              48.0|
|     Mme|         1|              24.0|
|    Capt|         1|              70.0|
|     Sir|         1|              49.0|
|Jonkheer|         1|              38.0|
|      Ms|         1|              28.0|
|     Col|         2|              58.0|
|    Mlle|         2|              24.0|
|   Major|         2|              48.5|
|     Rev|         6|43.166666666666664|
|      Dr|         6|              42.0|
|  Master|        36| 4.574166666666667|
|     Mrs|       108|35.898148148148145|
|    Miss|       146|21.773972602739725|
|      Mr|       398|32.368090452261306|
+--------+----------+------------------+



In [21]:
title_dic = {'Mr':'Mr', 'Miss':'Miss', 'Mrs':'Mrs', 'Master':'Master', \
             'Mlle': 'Miss', 'Major': 'Mr', 'Col': 'Mr', 'Sir': 'Mr',\
             'Don': 'Mr', 'Mme': 'Miss', 'Jonkheer': 'Mr', 'Lady': 'Mrs',\
             'Capt': 'Mr', 'Countess': 'Mrs', 'Ms': 'Miss', 'Dona': 'Mrs', \
             'Dr':'Mr', 'Rev':'Mr'}

In [22]:
mapping = create_map([lit(x) for x in chain(*title_dic.items())])
df1 = df1.withColumn('Title', mapping[df1['Title']])
df1.groupby('Title').mean('Age').show()

+------+------------------+
| Title|          avg(Age)|
+------+------------------+
|  Miss|             21.86|
|Master| 4.574166666666667|
|    Mr| 33.02272727272727|
|   Mrs|35.981818181818184|
+------+------------------+



### Извлечение сведений по возрасту

In [23]:
def age_imputer(df, title, age):
    return df.withColumn('Age', when((df['Age'].isNull()) & (df['Title'] == title), age).otherwise(df['Age']))


In [24]:
df1 = age_imputer(df1, 'Mr', 33.02)
df1 = age_imputer(df1, 'Miss', 21.86)
df1 = age_imputer(df1, 'Master', 4.57)
df1 = age_imputer(df1, 'Mrs', 35.98)

In [25]:
for col in df1.columns:
    print(col.ljust(20), df1.filter(df1[col].isNull()).count())

PassengerId          0
Survived             0
Pclass               0
Name                 0
Sex                  0
Age                  0
SibSp                0
Parch                0
Ticket               0
Fare                 0
Cabin                687
Embarked             2
Title                0


### Добавление агрегирующих данных и удаление неиспользуемых

In [26]:
df1 = df1.withColumn('FamilySize', df1['Parch'] + df1['Sibsp']).drop('Parch', 'Sibsp')
df1 = df1.drop('PassengerId', 'Cabin', 'Name', 'Ticket', 'Title')

In [27]:
df1.show(5)

+--------+------+------+----+-------+--------+----------+
|Survived|Pclass|   Sex| Age|   Fare|Embarked|FamilySize|
+--------+------+------+----+-------+--------+----------+
|       0|     3|  male|22.0|   7.25|       S|         1|
|       1|     1|female|38.0|71.2833|       C|         1|
|       1|     3|female|26.0|  7.925|       S|         0|
|       1|     1|female|35.0|   53.1|       S|         1|
|       0|     3|  male|35.0|   8.05|       S|         0|
+--------+------+------+----+-------+--------+----------+
only showing top 5 rows


In [28]:
df1 = df1.fillna({'Embarked' : 'S'})

In [29]:
for col in df1.columns:
    print(col.ljust(20), df1.filter(df1[col].isNull()).count())

Survived             0
Pclass               0
Sex                  0
Age                  0
Fare                 0
Embarked             0
FamilySize           0


### Построение модели (Spark ML)

- StringIndexer: преобразование строковых категорий в численные
- Векторный Ассемблер: Spark API
- Логистические регресии на основе регуляризации гребня и Лассо
- Древовидные ансамблевые методы: случайный лес
- GBT
- Конвейер

### Подключение библиотек из Spark ML

In [30]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

### StringIndexer

In [31]:
stringIndex = StringIndexer(inputCols=['Sex', 'Embarked'], outputCols=['SexNum', 'EmbarkedNum'])
stringIndex_model = stringIndex.fit(df1)
df1_ = stringIndex_model.transform(df1).drop('Sex', 'Embarked')
df1_.show(5)

+--------+------+----+-------+----------+------+-----------+
|Survived|Pclass| Age|   Fare|FamilySize|SexNum|EmbarkedNum|
+--------+------+----+-------+----------+------+-----------+
|       0|     3|22.0|   7.25|         1|   0.0|        0.0|
|       1|     1|38.0|71.2833|         1|   1.0|        1.0|
|       1|     3|26.0|  7.925|         0|   1.0|        0.0|
|       1|     1|35.0|   53.1|         1|   1.0|        0.0|
|       0|     3|35.0|   8.05|         0|   0.0|        0.0|
+--------+------+----+-------+----------+------+-----------+
only showing top 5 rows


### Векторный ассемблер

In [32]:
vec_asmbl = VectorAssembler(inputCols=df1_.columns[1:], outputCol='features')
df1_ = vec_asmbl.transform(df1_).select('features', 'Survived')
df1_.show(5, truncate=False)

+------------------------------+--------+
|features                      |Survived|
+------------------------------+--------+
|[3.0,22.0,7.25,1.0,0.0,0.0]   |0       |
|[1.0,38.0,71.2833,1.0,1.0,1.0]|1       |
|[3.0,26.0,7.925,0.0,1.0,0.0]  |1       |
|[1.0,35.0,53.1,1.0,1.0,0.0]   |1       |
|[3.0,35.0,8.05,0.0,0.0,0.0]   |0       |
+------------------------------+--------+
only showing top 5 rows


### Разбивка данных

In [33]:
train_df, test_df = df1_.randomSplit([0.7, 0.3])
train_df.show(5, truncate=False)

+---------------------+--------+
|features             |Survived|
+---------------------+--------+
|(6,[0,1],[1.0,33.02])|0       |
|(6,[0,1],[1.0,33.02])|0       |
|(6,[0,1],[1.0,38.0]) |0       |
|(6,[0,1],[2.0,33.02])|0       |
|(6,[0,1],[2.0,33.02])|0       |
+---------------------+--------+
only showing top 5 rows


In [38]:
train_df.show(30, truncate=False)

+--------------------------------+--------+
|features                        |Survived|
+--------------------------------+--------+
|(6,[0,1],[1.0,33.02])           |0       |
|(6,[0,1],[1.0,33.02])           |0       |
|(6,[0,1],[1.0,38.0])            |0       |
|(6,[0,1],[2.0,33.02])           |0       |
|(6,[0,1],[2.0,33.02])           |0       |
|(6,[0,1],[2.0,33.02])           |0       |
|(6,[0,1],[2.0,33.02])           |0       |
|(6,[0,1],[2.0,33.02])           |0       |
|(6,[0,1],[3.0,49.0])            |0       |
|[1.0,2.0,151.55,3.0,1.0,0.0]    |0       |
|[1.0,4.0,81.8583,2.0,0.0,0.0]   |1       |
|[1.0,11.0,120.0,3.0,0.0,0.0]    |1       |
|[1.0,14.0,120.0,3.0,1.0,0.0]    |1       |
|[1.0,15.0,211.3375,1.0,1.0,0.0] |1       |
|[1.0,16.0,39.4,1.0,1.0,0.0]     |1       |
|[1.0,16.0,57.9792,1.0,1.0,1.0]  |1       |
|[1.0,16.0,86.5,0.0,1.0,0.0]     |1       |
|[1.0,17.0,110.8833,2.0,0.0,1.0] |1       |
|[1.0,18.0,79.65,2.0,1.0,0.0]    |1       |
|[1.0,18.0,108.9,1.0,0.0,1.0]   

### Вычисления и метрики

In [40]:
evalutor = MulticlassClassificationEvaluator(labelCol='Survived', metricName='accuracy')

### Логистическая регрессия

In [41]:
logreg = LogisticRegression(labelCol='Survived', maxIter=100, elasticNetParam=0, regParam=0.3)
model = logreg.fit(train_df)
pred = model.transform(test_df)
evalutor.evaluate(pred)

0.792

### Случайный лес

In [49]:
rf = RandomForestClassifier(labelCol='Survived', numTrees=100, maxDepth=3)
model = rf.fit(train_df)
pred = model.transform(test_df)
evalutor.evaluate(pred)

0.816

### Градиентный бустинг

In [55]:
gb = GBTClassifier(labelCol='Survived', maxIter=100, maxDepth=2)
model = gb.fit(train_df)
pred = model.transform(test_df)
evalutor.evaluate(pred)

0.824

### Grid search CV

In [56]:
df1.show()

+--------+------+------+-----+-------+--------+----------+
|Survived|Pclass|   Sex|  Age|   Fare|Embarked|FamilySize|
+--------+------+------+-----+-------+--------+----------+
|       0|     3|  male| 22.0|   7.25|       S|         1|
|       1|     1|female| 38.0|71.2833|       C|         1|
|       1|     3|female| 26.0|  7.925|       S|         0|
|       1|     1|female| 35.0|   53.1|       S|         1|
|       0|     3|  male| 35.0|   8.05|       S|         0|
|       0|     3|  male|33.02| 8.4583|       Q|         0|
|       0|     1|  male| 54.0|51.8625|       S|         0|
|       0|     3|  male|  2.0| 21.075|       S|         4|
|       1|     3|female| 27.0|11.1333|       S|         2|
|       1|     2|female| 14.0|30.0708|       C|         1|
|       1|     3|female|  4.0|   16.7|       S|         2|
|       1|     1|female| 58.0|  26.55|       S|         0|
|       0|     3|  male| 20.0|   8.05|       S|         0|
|       0|     3|  male| 39.0| 31.275|       S|         

In [57]:
pipeline_rf = Pipeline(stages=[stringIndex, vec_asmbl, rf])

paramGrid = ParamGridBuilder().addGrid(rf.maxDepth, [3,4,5]).addGrid(rf.minInfoGain, [0, 0.01, 0.1]).addGrid(rf.numTrees, [100, 200, 500, 1000]).build()

selected_model = CrossValidator(estimator=pipeline_rf, estimatorParamMaps=paramGrid, evaluator=evalutor, numFolds=5)

model_final = selected_model.fit(df1)
pred = model_final.transform(df1)
evalutor.evaluate(pred)

26/09/24 16:18:33 WARN DAGScheduler: Broadcasting large task binary with size 1583.7 KiB
26/09/24 16:18:34 WARN DAGScheduler: Broadcasting large task binary with size 1088.0 KiB
26/09/24 16:18:38 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/09/24 16:18:42 WARN DAGScheduler: Broadcasting large task binary with size 1571.3 KiB
26/09/24 16:18:43 WARN DAGScheduler: Broadcasting large task binary with size 1070.8 KiB
26/09/24 16:18:46 WARN DAGScheduler: Broadcasting large task binary with size 3.0 MiB
26/09/24 16:18:50 WARN DAGScheduler: Broadcasting large task binary with size 1262.3 KiB
26/09/24 16:18:54 WARN DAGScheduler: Broadcasting large task binary with size 2.4 MiB
26/09/24 16:18:57 WARN DAGScheduler: Broadcasting large task binary with size 1150.4 KiB
26/09/24 16:19:00 WARN DAGScheduler: Broadcasting large task binary with size 1902.6 KiB
26/09/24 16:19:01 WARN DAGScheduler: Broadcasting large task binary with size 1088.0 KiB
26/09/24 16:19:01 WARN DAGSche

0.8507295173961841

In [58]:
best_model = model_final.bestModel

In [59]:
best_max_depth = best_model.stages[-1].getOrDefault('maxDepth')
best_min_info_gain = best_model.stages[-1].getOrDefault('minInfoGain')
best_num_trees = best_model.stages[-1].getOrDefault('numTrees')

print('maxDepth:', best_max_depth)
print('minInfoGain:', best_min_info_gain)
print('numTrees:', best_num_trees)

maxDepth: 5
minInfoGain: 0.0
numTrees: 500
